# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`


In [1]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90

In [2]:

import os
import getpass
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass(
        "NVIDIA_API_KEY: "
    ).strip()

if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

# Modelos alternativos indicados por el profesor
GENERATION_MODEL = "nvidia/nemotron-3-ultra-550b-a55b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

print("✅ Configuración NVIDIA NIM lista")
print("Modelo generativo:", GENERATION_MODEL)
print("Modelo embeddings:", EMBEDDING_MODEL)


✅ Configuración NVIDIA NIM lista
Modelo generativo: nvidia/nemotron-3-ultra-550b-a55b
Modelo embeddings: nvidia/nemotron-3-embed-1b


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


In [3]:
from markitdown import MarkItDown

DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return

    response = requests.get(url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")


for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")


⬇ Descargado: attention_paper.pdf (2.2 MB)
⬇ Descargado: rag_paper.pdf (0.9 MB)
✅ attention_paper.pdf → 40,174 caracteres
✅ rag_paper.pdf       → 69,121 caracteres

Inicio del paper de Attention:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog...


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)


Attention → 55 chunks
RAG       → 95 chunks
Total     → 150 chunks

Ejemplo de metadatos: {'title': 'Attention Is All You Need', 'source': 'attention_paper.pdf', 'chunk_index': 0}


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [5]:

import os
import requests

from langchain_core.embeddings import Embeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

# =====================================
# EMBEDDINGS NVIDIA
# =====================================

class NVIDIAEmbeddingsDirectos(Embeddings):

    def __init__(self, model):
        self.model = model

    def generar(self, textos, tipo):
        url = "https://integrate.api.nvidia.com/v1/embeddings"

        headers = {
            "Authorization": f"Bearer {os.environ['NVIDIA_API_KEY']}",
            "Content-Type": "application/json",
            "Accept": "application/json"
        }

        payload = {
            "model": self.model,
            "input": textos,
            "input_type": tipo,
            "truncate": "END",
            "encoding_format": "float"
        }

        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=120
        )

        if not response.ok:
            print("❌ Error de NVIDIA")
            print("Código:", response.status_code)
            print("Respuesta:", response.text[:1000])
            response.raise_for_status()

        datos = response.json()["data"]
        datos = sorted(datos, key=lambda x: x["index"])

        return [item["embedding"] for item in datos]

    def embed_query(self, text):
        return self.generar([text], "query")[0]

    def embed_documents(self, texts):
        vectores = []

        for i in range(0, len(texts), 5):
            lote = texts[i:i+5]
            vectores.extend(
                self.generar(lote, "passage")
            )

            print(
                f"Procesados {len(vectores)}/{len(texts)}",
                end="\r"
            )

        return vectores


# =====================================
# CONFIGURACIÓN DEL MODELO
# =====================================

embeddings = NVIDIAEmbeddingsDirectos(
    model=EMBEDDING_MODEL
)

print("Probando modelo:", EMBEDDING_MODEL)

EMBEDDING_DIM = len(
    embeddings.embed_query("dimension probe")
)

print("✅ Modelo NVIDIA funcionando")
print("Dimensión:", EMBEDDING_DIM)


# =====================================
# CREAR QDRANT
# =====================================

qdrant_client = QdrantClient(":memory:")

qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,
        distance=Distance.COSINE
    )
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings
)


# =====================================
# INDEXAR DOCUMENTOS
# =====================================

print("Indexando documentos...")

ids = vectorstore.add_documents(all_docs)

print()
print(f"✅ {len(ids)} chunks indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")
print("✅ PARTE 3 COMPLETADA")


Probando modelo: nvidia/nemotron-3-embed-1b
✅ Modelo NVIDIA funcionando
Dimensión: 2048
Indexando documentos...
Procesados 22/22
✅ 150 chunks indexados en Qdrant
✅ Dimensión de embeddings: 2048
✅ PARTE 3 COMPLETADA


---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [6]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

modelos = ChatNVIDIA.get_available_models()

print("Modelos de generación disponibles:")
for modelo in modelos:
    print(modelo.id)

Modelos de generación disponibles:
mistralai/mistral-7b-instruct-v0.2
nvidia/llama3-chatqa-1.5-8b
qwen/qwen3-next-80b-a3b-thinking
mistralai/mistral-nemotron
deepseek-ai/deepseek-v4-pro
mistralai/mamba-codestral-7b-v0.1
nvidia/nemotron-3-ultra-550b-a55b
microsoft/phi-4-mini-instruct
institute-of-science-tokyo/llama-3.1-swallow-70b-instruct-v0.1
nvidia/nvclip
microsoft/phi-3-mini-128k-instruct
deepseek-ai/deepseek-r1
qwen/qwen2.5-coder-32b-instruct
zyphra/zamba2-7b-instruct
mistralai/mistral-small-24b-instruct
tokyotech-llm/llama-3-swallow-70b-instruct-v0.1
bytedance/seed-oss-36b-instruct
deepseek-ai/deepseek-v4-flash
meta/llama-3.2-90b-vision-instruct
utter-project/eurollm-9b-instruct
marin/marin-8b-instruct
deepseek-ai/deepseek-r1-distill-qwen-14b
google/gemma-3n-e2b-it
google/gemma-2-9b-it
meta/llama-3.3-70b-instruct
moonshotai/kimi-k2-instruct-0905
moonshotai/kimi-k2.5
nvidia/llama-3.3-nemotron-super-49b-v1
meta/llama3-8b-instruct
mistralai/codestral-22b-instruct-v0.1
meta/llama-3.1

In [7]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=800,
)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts. "
        "If the excerpts do not contain enough evidence, say so explicitly. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = rag_chain.invoke(test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)


Pregunta: What is the Transformer architecture?
Respuesta: The Transformer is a neural network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. It uses an encoder-decoder structure with stacked self-attention and point-wise fully connected layers. The encoder consists of 6 identical layers, each containing a multi-head self-attention mechanism and a position-wise feed-forward network, with residual connections and layer normalization. All sub-layers and embeddings use dimension d=512. This architecture enables greater parallelization and achieves state-of-the-art results in machine translation.  

**Source:** *Attention Is All You Need*


---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [8]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

Q: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?
A: The Transformer (big model) achieved a BLEU score of **28.4** on the WMT 2014 English-to-German translation task.

This is stated in the paper: "On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big) in Table 2) outperforms the best previously reported models (including ensembles) by more than 2.0 BLEU, establishing a new state-of-the-art BLEU score of 28.4." [Attention Is All You Need | attention_paper.pdf]


<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [10]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

Q: How does RAG combine parametric and non-parametric memory?
A: RAG combines parametric and non-parametric memory by using a pre-trained seq2seq transformer (BART) as parametric memory and a dense vector index of Wikipedia accessed via a pre-trained neural retriever (DPR) as non-parametric memory. These components are integrated in a probabilistic model trained end-to-end: the retriever provides latent documents conditioned on the input, and the generator conditions on both the input and retrieved documents to produce the output. The latent documents are marginalized using a top-K approximation, either assuming the same document for the whole sequence (per-output) or allowing different documents per token (per-token).  

**Paper:** Retrieval-Augmented Generation (rag_paper.pdf)


<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [11]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

Q: What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?
A: The scaling factor \(1/\sqrt{d_k}\) prevents the dot products from growing too large in magnitude when \(d_k\) is large, which would push the softmax function into regions with extremely small gradients, hindering training. This is explained in the paper "Attention Is All You Need".


<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [12]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

Q: What are the two RAG model variants proposed in the paper and how do they differ?
A: The paper proposes two RAG model variants:

1. **RAG-Sequence**: Uses the **same retrieved document** for the entire generated sequence (marginalizes latent documents on a per-output basis).
2. **RAG-Token**: Can use **different retrieved documents for different tokens** (marginalizes latent documents on a per-token basis).

The key difference is whether retrieval is conditioned once per output sequence (RAG-Sequence) or per generated token (RAG-Token).

**Paper title**: *Retrieval-Augmented Generation* (rag_paper.pdf)


<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [13]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

Q: How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?
A: The base Transformer model uses **8 attention heads** (h = 8), with **key and value dimensions of 64 each per head** (d_k = d_v = d_model/h = 64).  
**Paper:** Attention Is All You Need


<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

In [14]:

# Evidencia de fuentes recuperadas para las 5 preguntas

preguntas = [q1, q2, q3, q4, q5]
respuestas = [answer1, answer2, answer3, answer4, answer5]

for i, (pregunta, respuesta) in enumerate(
    zip(preguntas, respuestas), start=1
):
    print("=" * 65)
    print(f"PREGUNTA {i}")
    print("Pregunta:", pregunta)
    print("\nRespuesta del RAG:", respuesta)

    documentos = retriever.invoke(pregunta)

    print("\nFuentes recuperadas:")
    for j, doc in enumerate(documentos, start=1):
        print(
            f"{j}. {doc.metadata.get('title', 'Sin título')}"
        )
        print(
            f"   Archivo: {doc.metadata.get('source', 'Desconocido')}"
        )
        print(
            f"   Chunk: {doc.metadata.get('chunk_index', 'N/A')}"
        )

    print("=" * 65, "\n")


PREGUNTA 1
Pregunta: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?

Respuesta del RAG: The Transformer (big model) achieved a BLEU score of **28.4** on the WMT 2014 English-to-German translation task.

This is stated in the paper: "On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big) in Table 2) outperforms the best previously reported models (including ensembles) by more than 2.0 BLEU, establishing a new state-of-the-art BLEU score of 28.4." [Attention Is All You Need | attention_paper.pdf]

Fuentes recuperadas:
1. Attention Is All You Need
   Archivo: attention_paper.pdf
   Chunk: 28
2. Attention Is All You Need
   Archivo: attention_paper.pdf
   Chunk: 1
3. Attention Is All You Need
   Archivo: attention_paper.pdf
   Chunk: 26
4. Attention Is All You Need
   Archivo: attention_paper.pdf
   Chunk: 40
5. Attention Is All You Need
   Archivo: attention_paper.pdf
   Chunk: 31

PREGUNTA 2
Pregunta: How d

## B5 — Quiz propio con Ground Truth

In [15]:

# ── Pregunta 6: Positional Encoding ──────────────

q6 = "Why does the Transformer use positional encodings, and which mathematical functions are used to generate them?"

answer6 = rag_chain.invoke(q6)

print(f"Q: {q6}")
print(f"A: {answer6}")


Q: Why does the Transformer use positional encodings, and which mathematical functions are used to generate them?
A: The Transformer uses positional encodings because the model contains no recurrence and no convolution, so it must inject information about the relative or absolute position of tokens in the sequence. The encodings are generated using sine and cosine functions of different frequencies:

PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

where pos is the position and i is the dimension. This choice was hypothesized to allow the model to easily learn to attend by relative positions. [Attention Is All You Need]



<details>
<summary>✅ Ground Truth — Pregunta 6</summary>

El Transformer utiliza **positional encodings** porque su arquitectura
no emplea recurrencia ni convoluciones y, por lo tanto, necesita
incorporar información sobre el orden de los tokens.

Para representar las posiciones, utiliza funciones **seno y coseno**
de diferentes frecuencias, cuyos valores se suman a los embeddings
de entrada del encoder y del decoder.

Estas representaciones permiten introducir información sobre
posiciones absolutas y facilitan el aprendizaje de relaciones
entre posiciones.

**Fuente:** Section 3.5 — Positional Encoding,
*Attention Is All You Need*.

</details>


In [16]:

# ── Pregunta 7: Retriever DPR ─────────────────

q7 = "How does the DPR retriever represent questions and documents to retrieve relevant information in RAG?"

answer7 = rag_chain.invoke(q7)

print(f"Q: {q7}")
print(f"A: {answer7}")


Q: How does the DPR retriever represent questions and documents to retrieve relevant information in RAG?
A: The DPR retriever uses a bi-encoder architecture where both questions and documents are encoded into dense vector representations using separate BERT-base encoders. Specifically, documents are encoded by a BERT document encoder to produce d(z), and questions are encoded by a BERT query encoder to produce q(x). Retrieval is performed by computing the inner product d(z)ᵀq(x) and finding the top-k documents via Maximum Inner Product Search (MIPS).

**Paper title:** Retrieval-Augmented Generation



<details>
<summary>✅ Ground Truth — Pregunta 7</summary>

El sistema RAG utiliza **Dense Passage Retrieval (DPR)** para
recuperar documentos relevantes.

DPR emplea una arquitectura **bi-encoder**, con un encoder para
representar las preguntas y otro para representar los documentos.

Ambos convierten los textos en vectores densos. Después,
el sistema compara sus representaciones mediante productos
internos y recupera los documentos con mayor puntuación
de similitud.

Estos documentos forman parte de la **memoria no paramétrica**
y proporcionan información adicional al modelo generativo.

**Fuente:** Section 2.2 — Retriever: DPR,
*Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*.

</details>


In [17]:

# ── Pregunta 8: Transformer + RAG ──────────────

q8 = (
    "How does the Transformer encoder-decoder attention mechanism "
    "help explain how RAG uses retrieved documents to generate "
    "context-aware answers?"
)

answer8 = rag_chain.invoke(q8)

print(f"Q: {q8}")
print(f"A: {answer8}")


Q: How does the Transformer encoder-decoder attention mechanism help explain how RAG uses retrieved documents to generate context-aware answers?
A: The provided excerpts do not contain enough evidence to explain how the Transformer encoder-decoder attention mechanism specifically helps RAG use retrieved documents for context-aware generation.

While the RAG paper states that its generator (BART, a seq2seq transformer) "conditions on these latent documents together with the input to generate the output" and the Attention paper describes how encoder-decoder attention "allows every position in the decoder to attend over all positions in the input sequence," neither excerpt explicitly connects these mechanisms or explains how the attention mechanism enables RAG's document conditioning.

[Retrieval-Augmented Generation | rag_paper.pdf] and [Attention Is All You Need | attention_paper.pdf]



<details>
<summary>✅ Ground Truth — Pregunta 8 (Ambos papers)</summary>

El artículo **Attention Is All You Need** presenta el mecanismo
de atención del Transformer, que permite al decoder utilizar
información de las representaciones producidas por el encoder.

En la atención encoder-decoder, las queries provienen del
decoder, mientras que las keys y values proceden del encoder.
Esto permite que el modelo atienda a información relevante
durante la generación de la secuencia.

Por otro lado, el artículo de **RAG** propone recuperar
documentos externos mediante DPR y proporcionarlos como
contexto adicional a un generador seq2seq basado en
Transformer, específicamente BART-large.

La relación entre ambos artículos es que RAG incorpora
información recuperada a un generador basado en la
arquitectura Transformer, cuyos mecanismos de atención
permiten utilizar el contexto para producir una respuesta.

La recuperación externa y la atención cumplen funciones
complementarias: la primera aporta información y la segunda
ayuda al generador a utilizarla.

**Fuentes:**
- Section 3.2.3 — Applications of Attention in our Model,
  *Attention Is All You Need*.
- Sections 2.1, 2.2 y 2.3 — RAG Models, Retriever y Generator,
  *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*.

</details>


In [18]:

# ==========================================
# B5 - EVALUACIÓN DEL QUIZ PROPIO
# ==========================================

preguntas_b5 = [q6, q7, q8]
respuestas_b5 = [answer6, answer7, answer8]

for i, (pregunta, respuesta) in enumerate(
    zip(preguntas_b5, respuestas_b5),
    start=6
):
    print("=" * 65)
    print(f"PREGUNTA {i}")
    print("Pregunta:", pregunta)

    print("\nRESPUESTA DEL RAG:")
    print(respuesta)

    documentos = retriever.invoke(pregunta)

    print("\nFUENTES RECUPERADAS:")

    fuentes = set()

    for j, doc in enumerate(documentos, start=1):
        titulo = doc.metadata.get("title", "Sin título")
        archivo = doc.metadata.get("source", "Desconocido")
        chunk = doc.metadata.get("chunk_index", "N/A")

        fuentes.add(archivo)

        print(f"{j}. {titulo}")
        print(f"   Archivo: {archivo}")
        print(f"   Chunk: {chunk}")

    if i == 8:
        ambos = {
            "attention_paper.pdf",
            "rag_paper.pdf"
        }.issubset(fuentes)

        print("\n¿Se recuperaron ambos papers?")
        print("Sí" if ambos else "No")

    print("=" * 65, "\n")


PREGUNTA 6
Pregunta: Why does the Transformer use positional encodings, and which mathematical functions are used to generate them?

RESPUESTA DEL RAG:
The Transformer uses positional encodings because the model contains no recurrence and no convolution, so it must inject information about the relative or absolute position of tokens in the sequence. The encodings are generated using sine and cosine functions of different frequencies:

PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

where pos is the position and i is the dimension. This choice was hypothesized to allow the model to easily learn to attend by relative positions. [Attention Is All You Need]

FUENTES RECUPERADAS:
1. Attention Is All You Need
   Archivo: attention_paper.pdf
   Chunk: 19
2. Attention Is All You Need
   Archivo: attention_paper.pdf
   Chunk: 18
3. Attention Is All You Need
   Archivo: attention_paper.pdf
   Chunk: 16
4. Attention Is All You Need
   Archivo: attention_

# Análisis individual del quiz RAG

## Pregunta 1 — BLEU score del Transformer

**Pregunta:**

What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?

**Respuesta obtenida del RAG:**

El modelo Transformer big obtuvo un BLEU score de **28.4** en la tarea de traducción WMT 2014 English-to-German.

El sistema también señaló que el resultado fue reportado en el artículo *Attention Is All You Need*.

**Ground truth:**

El modelo Transformer big obtuvo 28.4 BLEU, superando los mejores resultados previos por más de dos puntos.

**Fragmentos recuperados:**

- Artículo: *Attention Is All You Need*.
- Archivo: `attention_paper.pdf`.
- Chunks: 28, 1, 26, 40 y 31.

**Evaluación:**

- Correctitud: **Sí**. Identificó correctamente el valor 28.4.
- Fuente: **Correcta**. Los cinco fragmentos provinieron del artículo esperado.
- Completitud: **Sí**. Respondió la pregunta e identificó la tarea de traducción.

**Observación:** el modelo incluyó el dato central solicitado y lo respaldó mediante referencias al contenido recuperado. No mencionó expresamente la ventaja de más de dos puntos BLEU frente a los resultados anteriores, pero ese detalle no era indispensable para responder la pregunta.

---

## Pregunta 2 — Memoria paramétrica y no paramétrica

**Pregunta:**

How does RAG combine parametric and non-parametric memory?

**Respuesta obtenida del RAG:**

RAG combina una memoria paramétrica representada por el modelo generativo BART con una memoria no paramétrica formada por un índice vectorial de documentos de Wikipedia.

La recuperación se realiza mediante DPR, que proporciona documentos relevantes al generador.

El modelo integra ambas fuentes de información mediante un enfoque probabilístico y considera los documentos recuperados durante la generación.

**Ground truth:**

RAG combina BART-large, como memoria paramétrica, con un índice denso de Wikipedia, como memoria no paramétrica. DPR recupera documentos relevantes y el generador utiliza esa información para producir respuestas mediante un sistema entrenado conjuntamente.

**Fragmentos recuperados:**

- Artículo: *Retrieval-Augmented Generation*.
- Archivo: `rag_paper.pdf`.
- Chunks: 1, 8, 32, 7 y 50.

**Evaluación:**

- Correctitud: **Sí**. Diferenció correctamente ambos tipos de memoria.
- Fuente: **Correcta**. Todos los fragmentos provinieron del artículo RAG.
- Completitud: **Sí**. Explicó la relación entre BART, DPR, Wikipedia y el proceso generativo.

**Observación:** la respuesta fue detallada e incorporó información adicional sobre la marginalización de documentos y las variantes de generación.

---

## Pregunta 3 — Factor de escala de la atención

**Pregunta:**

What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?

**Respuesta obtenida del RAG:**

El factor de escala \(1/\sqrt{d_k}\) evita que los productos punto alcancen magnitudes demasiado elevadas cuando aumenta la dimensión de las claves.

Esto impide que la función softmax entre en regiones con gradientes extremadamente pequeños y dificulte el entrenamiento.

**Ground truth:**

El escalamiento por \(\sqrt{d_k}\) reduce la magnitud de los productos punto, evita la saturación de softmax y favorece un entrenamiento más estable.

**Fragmentos recuperados:**

- Artículo: *Attention Is All You Need*.
- Archivo: `attention_paper.pdf`.
- Chunks: 11, 12, 13, 16 y 20.

**Evaluación:**

- Correctitud: **Sí**. Explicó correctamente el propósito del factor de escala.
- Fuente: **Correcta**. Todos los fragmentos procedieron del artículo Transformer.
- Completitud: **Sí**. Incluyó la relación entre magnitud, softmax y gradientes.

**Observación:** la explicación coincidió con la justificación teórica de Scaled Dot-Product Attention.

---

## Pregunta 4 — Variantes RAG-Sequence y RAG-Token

**Pregunta:**

What are the two RAG model variants proposed in the paper and how do they differ?

**Respuesta obtenida del RAG:**

El modelo identificó las dos variantes propuestas:

**RAG-Sequence:** utiliza un mismo documento recuperado para condicionar la generación de toda la secuencia.

**RAG-Token:** permite utilizar documentos recuperados diferentes durante la generación de distintos tokens.

La diferencia principal se encuentra en cómo se consideran los documentos durante el proceso generativo.

**Ground truth:**

RAG-Sequence utiliza un documento recuperado como variable latente para toda la secuencia, mientras que RAG-Token permite considerar documentos diferentes por token, ofreciendo mayor flexibilidad.

**Fragmentos recuperados:**

- Artículo: *Retrieval-Augmented Generation*.
- Archivo: `rag_paper.pdf`.
- Chunks: 10, 11, 1, 50 y 7.

**Evaluación:**

- Correctitud: **Sí**. Identificó las dos variantes y sus diferencias.
- Fuente: **Correcta**. Recuperó únicamente fragmentos del paper RAG.
- Completitud: **Sí**. Describió cómo cambia el condicionamiento de la generación.

**Observación:** la respuesta fue correcta y suficientemente detallada. No mencionó el mayor costo computacional de RAG-Token, que aparece como información complementaria en el ground truth.

---

## Pregunta 5 — Cabezas de atención del Transformer

**Pregunta:**

How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?

**Respuesta obtenida del RAG:**

El Transformer base utiliza **8 cabezas de atención**, con dimensiones de claves y valores de **64 por cabeza**.

El modelo también presentó la relación:

\[
d_k=d_v=\frac{d_{model}}{h}=\frac{512}{8}=64
\]

**Ground truth:**

El modelo base utiliza 8 cabezas en paralelo, con dimensiones de clave y valor iguales a 64.

**Fragmentos recuperados:**

- Artículo: *Attention Is All You Need*.
- Archivo: `attention_paper.pdf`.
- Chunks: 14, 15, 9, 0 y 40.

**Evaluación:**

- Correctitud: **Sí**. Identificó los valores 8, 64 y 64.
- Fuente: **Correcta**. Los fragmentos procedieron del artículo correspondiente.
- Completitud: **Sí**. Incluyó los valores y la relación matemática entre las dimensiones.

**Observación:** la respuesta resolvió correctamente la pregunta e incluyó la fórmula utilizada para obtener la dimensión por cabeza.

---

## Conclusión del quiz

Las cinco respuestas obtenidas mediante el pipeline RAG fueron correctas respecto a los datos principales de referencia.

En todos los casos, los cinco fragmentos recuperados pertenecieron al artículo académico esperado, lo que evidencia un funcionamiento adecuado de la recuperación semántica para las consultas realizadas.

Las respuestas mostraron distintos niveles de detalle, pero ninguna omitió la información indispensable para resolver su respectiva pregunta.

Por lo tanto, la evaluación del quiz fue satisfactoria, tanto en correctitud como en correspondencia de fuentes y completitud esencial.

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [ ]:

# BONUS - Verificar qué papers recupera Qdrant

docs_bonus = retriever.invoke(q_bonus)

print("FUENTES RECUPERADAS:")

for i, doc in enumerate(docs_bonus, 1):
    print(f"{i}. {doc.metadata.get('title')}")
    print(f"   Archivo: {doc.metadata.get('source')}")
    print(f"   Chunk: {doc.metadata.get('chunk_index')}")

papers = {doc.metadata.get("source") for doc in docs_bonus}

print("\n¿Aparecieron ambos papers?")
print("Sí" if {"rag_paper.pdf", "attention_paper.pdf"}.issubset(papers) else "No")


In [ ]:

# BONUS - Generar respuesta con los fragmentos ya recuperados

contexto_bonus = format_docs(docs_bonus)

cadena_bonus = prompt | llm | StrOutputParser()

try:
    respuesta_bonus = cadena_bonus.invoke({
        "context": contexto_bonus,
        "question": q_bonus
    })

    print("PREGUNTA BONUS:")
    print(q_bonus)

    print("\nRESPUESTA DEL RAG:")
    print(respuesta_bonus)

except Exception as error:
    print("La generación no se completó:")
    print(type(error).__name__, str(error)[:500])


<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.


# Reflexión final

### 1. ¿Qué ocurrió cuando el contexto no contenía información suficiente? ¿El modelo reconoció la limitación o inventó una respuesta?

Durante las cinco preguntas del quiz, el sistema logró recuperar fragmentos relevantes y generar respuestas correctas, por lo que no se presentó un caso evidente de falta de información.

Sin embargo, el prompt del sistema está configurado para que el modelo responda únicamente utilizando el contexto recuperado y reconozca cuando no existe evidencia suficiente. Esto busca evitar las alucinaciones, es decir, respuestas inventadas o que no están respaldadas por los documentos.

Para comprobar si realmente cumple esta instrucción, sería necesario realizar una pregunta cuya respuesta no aparezca en ninguno de los dos artículos. Por lo tanto, con las pruebas realizadas no es posible afirmar que el modelo siempre reconocerá sus limitaciones.

### 2. ¿Qué fragmentos recuperó para la pregunta bonus? ¿Aparecieron ambos papers?

La pregunta bonus busca relacionar el modelo generativo utilizado en RAG con la arquitectura Transformer presentada en *Attention Is All You Need*.

La respuesta de referencia indica que RAG utiliza **BART-large** como modelo generativo, el cual está basado en la arquitectura Transformer.

En las cinco preguntas anteriores, el recuperador seleccionó correctamente cinco fragmentos del artículo correspondiente a cada consulta. Sin embargo, no se cuenta todavía con la salida específica de los fragmentos recuperados para la pregunta bonus, por lo que no es posible confirmar si aparecieron ambos artículos.

Para verificarlo, se deben revisar los títulos y archivos mostrados por la celda bonus. Lo ideal sería que el sistema recuperara información de *Retrieval-Augmented Generation* y *Attention Is All You Need*, aunque una respuesta correcta también podría construirse con fragmentos del primer artículo si contienen la información necesaria.

### 3. ¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?

Estos parámetros influyen directamente en la información recuperada y, por lo tanto, en la calidad de las respuestas generadas.

- **`chunk_size`:** determina el tamaño de los fragmentos. Si aumenta, cada fragmento contiene más contexto, pero también puede incorporar información innecesaria. Si disminuye, la búsqueda puede ser más específica, aunque existe el riesgo de dividir explicaciones importantes.

- **`chunk_overlap`:** establece cuánto contenido se comparte entre fragmentos consecutivos. Un mayor solapamiento ayuda a conservar la continuidad de las ideas, pero aumenta la redundancia y el número de fragmentos. Un solapamiento menor reduce esa redundancia, aunque podría separar información relacionada.

- **`k`:** indica el número máximo de fragmentos que recupera el sistema. Aumentarlo puede proporcionar más evidencia, pero también incorporar información poco relevante y aumentar el consumo de tokens. Disminuirlo reduce el contexto y los costos, aunque podría dejar información importante por fuera.

- **`score_threshold`:** establece la similitud mínima requerida para recuperar un fragmento. Un valor mayor permite filtrar resultados menos relacionados, pero puede descartar información útil. Un valor menor recupera más contenido, aunque aumenta el riesgo de incluir fragmentos irrelevantes.

En nuestro sistema utilizamos `chunk_size=900`, `chunk_overlap=120`, `k=5` y `score_threshold=0.25`. Estos valores permitieron recuperar fragmentos adecuados para las cinco preguntas evaluadas, pero sería útil experimentar con diferentes configuraciones para identificar cuáles ofrecen mejores resultados.

### 4. ¿Por qué conviene separar el modelo de embeddings del modelo generativo?

Conviene separarlos porque cumplen funciones diferentes dentro del sistema RAG.

El **modelo de embeddings** transforma los textos y las preguntas en representaciones vectoriales, permitiendo identificar fragmentos semánticamente relacionados. En nuestro caso, utilizamos `nvidia/nemotron-3-embed-1b`, que generó vectores de 2048 dimensiones.

El **modelo generativo** recibe la pregunta y los fragmentos recuperados para construir una respuesta en lenguaje natural. En nuestro caso, utilizamos `nvidia/nemotron-3-ultra-550b-a55b`.

Esta separación permite optimizar cada componente de manera independiente, elegir modelos especializados para cada tarea y controlar mejor los costos. También facilita actualizar el modelo generativo sin necesidad de volver a indexar los documentos, siempre que se conserve el mismo modelo y configuración de embeddings.

### 5. ¿Qué cambiarías para llevar este prototipo a producción? Considera persistencia, control de acceso, evaluación, observabilidad y costos.

Para llevar el prototipo a un entorno de producción realizaría varios cambios que permitan mejorar su seguridad, estabilidad y eficiencia.

**Persistencia:** actualmente Qdrant funciona en memoria, por lo que la información indexada se pierde al reiniciar la sesión. Utilizaría una base vectorial persistente para conservar los documentos y evitar repetir el proceso de indexación.

**Control de acceso:** implementaría autenticación y autorización para asegurar que cada usuario pueda consultar únicamente los documentos a los que tiene permiso. También protegería las API Keys mediante un gestor de secretos.

**Evaluación:** ampliaría las pruebas utilizando preguntas correctas, ambiguas y sin respuesta en los documentos. Evaluaría la precisión de las respuestas, la relevancia de los fragmentos recuperados y la presencia de posibles alucinaciones.

**Observabilidad:** incorporaría registros y métricas para monitorear los tiempos de respuesta, los errores de la API, los documentos recuperados y el comportamiento del modelo. Esto permitiría identificar problemas y mejorar continuamente el sistema.

**Costos:** optimizaría la cantidad de fragmentos recuperados, el tamaño del contexto y el número de solicitudes a NVIDIA. También implementaría mecanismos de caché para evitar generar embeddings repetidos y establecería límites de uso.

En conclusión, el prototipo demostró que es posible combinar recuperación semántica y generación de texto para consultar artículos académicos. Para utilizarlo en producción sería necesario complementar su funcionamiento con persistencia, seguridad, evaluación continua y mecanismos de monitoreo y optimización de recursos.